<a href="https://colab.research.google.com/github/ankorn/arcanumsearch/blob/main/arc_synth.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import random
import re
import json
from typing import List, Dict, Optional
from dataclasses import dataclass
import time
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [2]:
model_name = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            trust_remote_code=True
        )
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [21]:
@dataclass
class QueryExample:
    query: str
    query_type: str  # 'informational', 'navigational', 'transactional', 'vague'
    difficulty: str  # 'easy', 'medium', 'hard'

class LLMQueryGenerator:
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer

    def _generate(self, prompt: str, max_tokens: int = 512, temperature: float = 0.8) -> str:
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)

        outputs = self.model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
            pad_token_id=self.tokenizer.eos_token_id,
            stop_strings=["]"],
            tokenizer=self.tokenizer
        )

        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        # Remove the prompt from response
        response = response[len(prompt):].strip()
        return response

    def _create_query_generation_prompt(self,
                                        document_title: str,
                                        document_text: str,
                                        num_queries: int = 3) -> str:

        return f"""You are a player of the RPG game "Arcanum: Of Steamworks and Magick Obscura".
You need help with a quest and are searching a wiki for information.

Given this quest information:
Title: {document_title.split(': ')[0]}
Content: {document_text}...

Generate {num_queries} different search queries that a player might type to find this information.
Make them diverse in style and specificity:

Requirements:
1. Mix of specific (quest name) and vague (problem description) queries
2. Include natural language questions and keyword searches
3. Some should be short (2-3 words), others longer full sentences
4. Include common typos or variations a player might use
5. Some should focus on main result of a quest, others on walkthrough steps
6. Avoid using words "arcanum", "experience", "reward"

Output format: Return ONLY a JSON array of exactly {num_queries} strings, like:
["query 1", "query 2", "query 3"]

Queries:"""

    def generate_queries_for_document(self,
                                      title: str,
                                      text: str,
                                      num_queries: int = 3) -> List[QueryExample]:
        prompt = self._create_query_generation_prompt(title, text, num_queries)
        response = self._generate(prompt, temperature=0.9)

        try:
            # Parse JSON response
            queries = json.loads(response)
            if not isinstance(queries, list):
                queries = [str(queries)]
        except json.JSONDecodeError:
            # Fallback: extract lines that look like queries
            lines = [l.strip().strip('"\'') for l in response.split('\n') if l.strip()]
            queries = [l for l in lines if len(l) > 5 and not l.startswith('{')][:num_queries]

        examples = []
        for i, q in enumerate(queries[:3]):
            # Classify query type
            q_lower = q.lower()
            if any(w in q_lower for w in ['how', 'what', 'where', 'who', 'why']):
                q_type = 'informational'
            elif any(w in q_lower for w in ['find', 'get', 'obtain', 'kill', 'deliver']):
                q_type = 'transactional'
            else:
                q_type = 'navigational'

            # Estimate difficulty
            if len(q.split()) <= 3:
                difficulty = 'easy'
            elif any(w in q_lower for w in title.lower().split()):
                difficulty = 'easy'
            else:
                difficulty = 'medium'

            examples.append(QueryExample(
                query=q,
                query_type=q_type,
                difficulty=difficulty,
            ))

        return examples

class ArcanumSyntheticDataset:
    def __init__(self, csv_path: str, llm_generator: Optional[LLMQueryGenerator] = None):
        self.df = pd.read_csv(csv_path)
        self.generator = llm_generator

    def generate(self,
                 queries_per_doc: int = 3,
                 output_path: str = 'synthetic_queries_llm.csv') -> pd.DataFrame:

        results = []

        for idx, row in self.df.iterrows():
            print(f"Processing {idx+1}/{len(self.df)}: {row['TITLE'][:50]}...")

            if self.generator:
                examples = self.generator.generate_queries_for_document(
                    row['TITLE'], row['TEXT'], queries_per_doc
                )
                for ex in examples:
                    results.append({
                        'query': ex.query,
                        'document_title': row['TITLE'],
                        'document_text': row['TEXT'],
                        'document_link': row['LINK'],
                        'query_type': ex.query_type,
                        'difficulty': ex.difficulty
                    })

        df = pd.DataFrame(results)
        df.to_csv(output_path, index=False)
        print(f"\nGenerated {len(df)} examples, saved to {output_path}")
        return df

    def _create_row(self, row, query: str, label: int, label_type: str) -> Dict:
        return {
            'query': query,
            'document_title': row['TITLE'],
            'document_text': row['TEXT'],
            'document_link': row['LINK'],
            'label': label,
            'label_type': label_type
        }

In [22]:
csv_path = 'https://raw.githubusercontent.com/ankorn/nlp_course/refs/heads/2023/arcanum_quests.csv'

generator = LLMQueryGenerator(
    model,
    tokenizer
)

dataset = ArcanumSyntheticDataset(csv_path, generator)
df = dataset.generate(
    queries_per_doc=3,
    output_path='arcanum_queries_qwen.csv'
)

df

Processing 1/229: Find the Bow of Ecclesiastes: Walkthrough...
Processing 2/229: Find the Bow of Ecclesiastes: Notes...
Processing 3/229: Kill Sir Garrick Stout: Path 1, Dodge Mastery only...
Processing 4/229: Find Lady Druella: Path 2, Melee Mastery only...
Processing 5/229: Retrieve Azram's Star: Walkthrough...
Processing 6/229: Find the Master of Backstab: Short Description...
Processing 7/229: Run Around Tarant in Your Underwear: Short Descrip...
Processing 8/229: Find the Master of Prowling: Short Description...
Processing 9/229: Get staff of K’an T’au: Short Description...
Processing 10/229: Gamble with Gurin Rockharrow: Background...
Processing 11/229: Gamble with Gurin Rockharrow: Walkthrough...
Processing 12/229: Gamble with Gurin Rockharrow: Notes...
Processing 13/229: Acquire Ten Thousand Gold Pieces: Short Descriptio...
Processing 14/229: Find the Master of Heal: Short Description...
Processing 15/229: Negotiations with Caladon: Walkthrough...
Processing 16/229: Negotiation

,query,document_title,document_text,document_link,query_type,difficulty
0,Bow of Ecclesiastes location,Find the Bow of Ecclesiastes: Walkthrough,Black Root Kietzel Pierce can be found in Blac...,https://arcanum.fandom.com/wiki/Find_the_Bow_o...,navigational,easy
1,Kietzel mushroom inn night,Find the Bow of Ecclesiastes: Walkthrough,Black Root Kietzel Pierce can be found in Blac...,https://arcanum.fandom.com/wiki/Find_the_Bow_o...,navigational,medium
2,Ruins of Szabo undead guards,Find the Bow of Ecclesiastes: Walkthrough,Black Root Kietzel Pierce can be found in Blac...,https://arcanum.fandom.com/wiki/Find_the_Bow_o...,navigational,easy
3,Bow of Ecclesiastes location,Find the Bow of Ecclesiastes: Notes,This quest can be done in tandem with the Find...,https://arcanum.fandom.com/wiki/Find_the_Bow_o...,navigational,easy
4,Find Dudley Crosston exp,Find the Bow of Ecclesiastes: Notes,This quest can be done in tandem with the Find...,https://arcanum.fandom.com/wiki/Find_the_Bow_o...,transactional,easy
...,...,...,...,...,...,...
682,how to defeat arronax,Kill the Banished: Short Description,"144 Kerghan: Kill Arronax, The Bane of Kree, G...",https://arcanum.fandom.com/wiki/Kill_the_Banished,informational,medium
683,where is arronax located,Kill the Banished: Short Description,"144 Kerghan: Kill Arronax, The Bane of Kree, G...",https://arcanum.fandom.com/wiki/Kill_the_Banished,informational,medium
684,Falchion on island 5,Retrieve Kryggird's Falchion: Short Description,Offered by The Bane of Kree to find Kryggird's...,https://arcanum.fandom.com/wiki/Retrieve_Krygg...,navigational,medium
685,Where to find Kryggird's Falchion,Retrieve Kryggird's Falchion: Short Description,Offered by The Bane of Kree to find Kryggird's...,https://arcanum.fandom.com/wiki/Retrieve_Krygg...,informational,easy


In [20]:
import huggingface_hub
huggingface_hub.login()

In [23]:
from datasets import Dataset

Dataset.from_pandas(df).push_to_hub("pameydorke/arcanum-quests-queries-synthetic")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########|  118kB /  118kB            

CommitInfo(commit_url='https://huggingface.co/datasets/pameydorke/arcanum-quests-queries-synthetic/commit/0395d6f921f4a7cd887f2b00e8d883dcd18d31b8', commit_message='Upload dataset', commit_description='', oid='0395d6f921f4a7cd887f2b00e8d883dcd18d31b8', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/pameydorke/arcanum-quests-queries-synthetic', endpoint='https://huggingface.co', repo_type='dataset', repo_id='pameydorke/arcanum-quests-queries-synthetic'), pr_revision=None, pr_num=None)